<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/kikim6114/nlp2026/blob/main/python_tutorial/00-9.PyTorch_Tutorial2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

### 데모: Word Window Classification: 단어 단위 문맥 분류

문장 속 하나의 단어만을 보는 것으로는 부족하기 때문에,  
**단어 주변의 문맥(context)**을 함께 고려해야 한다.  
이를 위해 **Word Window Classification**이라는 고전적인 기법을 사용하여  
**단어 단위로 위치 정보(Location)**를 분류하는 문제를 해결한다.

---

### 문제 정의

- **목표**: 문장에서 각 단어가 `LOCATION`인지 아닌지를 판단하는 모델 생성
- **가정**: `LOCATION`은 항상 **한 단어짜리**만 존재 (예: `Paris`는 OK, `San Francisco`는 X)

---

### 데이터 만들기

- 각 단어에 대해 `"앞 단어, 현재 단어, 뒤 단어"`로 이루어진 **윈도우(window)** 생성
- 각 윈도우는 **하나의 입력**으로 사용되며,  
  **중앙 단어의 레이블**(LOCATION 여부)을 예측

---

### 모델링

- **입력**: 윈도우 내 단어들의 벡터(임베딩 또는 원-핫)
- **출력**: 중앙 단어가 `LOCATION`인지 아닌지를 판별하는 **이진 분류(binary classification)**

---

### 훈련

- **손실 함수**:
  - `nn.BCELoss()` 또는 `nn.CrossEntropyLoss()`  
- **옵티마이저**:
  - `Adam`, `SGD` 등

---

### 예측

- 문장을 입력으로 받아서  
  각 단어가 **LOCATION**인지 여부를 **순서대로 판단**


### Data

모든 머신 러닝 프로젝트의 첫 번째 작업은 훈련 세트를 설정하는 것이다. 일반적으로 사용할 훈련 코퍼스가 있을 것이다. NLP 작업에서 코퍼스는 일반적으로 각 행이 문장이나 테이블 형식 데이터 포인트에 해당하는 `.txt` 또는 `.csv` 파일이다. 이 간단한 작업에서는 데이터와 해당 레이블을 이미 `Python` 리스트로 읽었다고 가정하겠다.

In [ ]:
# Our raw data, which consists of sentences
corpus = [
          "We always come to Paris",
          "The professor is from Australia",
          "I live in Stanford",
          "He comes from Taiwan",
          "The capital of Turkey is Ankara"
         ]

### 전처리(Preprocessing)의 중요성

분석의 목적에 따라 어떤 전처리 과정을 수행할지가 결정되므로,  
**전처리 과정은 자연어 처리에서 필수적인 단계**이다.

---

### 주요 전처리 단계

1. **Tokenization (토크나이징)**  
   - 문장을 **단어 단위로 분리**  
   - 예: `"I love Paris"` → `["I", "love", "Paris"]`

2. **Lowercasing (소문자 변환)**  
   - 대소문자 구분 없이 처리하기 위해 **모든 단어를 소문자**로 변환  
   - 예: `"Paris"` → `"paris"`

3. **Noise removal (노이즈 제거)**  
   - 모델 학습에 방해되는 **특수문자, 기호, 이모지** 등을 제거  
   - 예: `"hello! 😊"` → `"hello"`

4. **Stop words removal (불용어 제거)**  
   - 자주 등장하지만 **정보량이 적은 단어들** 제거  
   - 예: `a`, `the`, `is`, `in`, `on`, ...

---

이러한 전처리 과정을 통해 **데이터의 품질을 높이고, 모델 성능 향상**에 기여할 수 있다.


In [ ]:
# 훈련 예제를 생성하기 위해 사용할 전처리 함수
# 이 함수는 간단하게 글자를 소문자로 바꾸고 단어를 토큰화한다.
def preprocess_sentence(sentence):
  return sentence.lower().split() #띄어쓰기 기준으로 구분 

# Create our training set
train_sentences = [preprocess_sentence(sent) for sent in corpus]
train_sentences

각 훈련 예제마다 해당 레이블도 있어야 한다. 모델의 목표가 어떤 단어가 `LOCATION`에 해당하는지 결정하는 것임을 상기하라. 즉, 모델이 `LOCATION`이 아닌 모든 단어에 대해 `0`을 출력하고, `LOCATION`인 단어에 대해 `1`을 출력하기를 원한다.

In [ ]:
# Set of locations that appear in our corpus
locations = set(["australia", "ankara", "paris", "stanford", "taiwan", "turkey"])

# Our train labels
train_labels = [[1 if word in locations else 0 for word in sent] for sent in train_sentences]
train_labels

### Converting Words to Embeddings (단어를 임베딩으로 변환)

우리의 입력은 `"paris"`, `"live"`처럼 **문자열 형태의 단어**이지만,  
머신러닝/딥러닝 모델은 **숫자 벡터**로만 연산이 가능하기 때문에  
**문자열을 숫자로 변환**하는 과정이 필요한다.

---

임베딩 조회 테이블 `E`가 있다고 상상해 보자. 여기서 각 행은 임베딩에 해당한다. 즉, 어휘에 있는 각 단어는 이 테이블에서 해당하는 임베딩 행 `i`를 갖습니다. 단어의 임베딩을 찾으려면 다음 단계를 따른다:
1. 임베딩 테이블에서 단어의 해당 인덱스 `i`를 찾는다: `word->index`.
2. 임베딩 테이블에 인덱싱하여 임베딩을 얻는다: `index->embedding`.

첫 번째 단계를 살펴보자. 어휘에 있는 모든 단어에 해당 인덱스를 지정해야 한다. 다음과 같이 할 수 있다:
1. 코퍼스에서 모든 고유 단어를 찾는다.
2. 각 단어에 인덱스를 지정한다.

In [ ]:
# Find all the unique words in our corpus
vocabulary = set(w for s in train_sentences for w in s)
vocabulary

### Out-of-Vocabulary (OOV) 단어 처리

현재의 **vocabulary**(어휘 사전)는 우리가 가진 말뭉치(corpus)에 등장한 모든 단어들을 포함하고 있다.  
하지만 **테스트 시점**(test time)에는 **어휘 사전에 없는 단어들**(out-of-vocabulary, OOV)을 마주칠 수 있다.

---

### 왜 OOV 처리가 필요한가?

모델은 예측 시에 **중심 단어 뿐만 아니라 주변 단어들의 문맥**도 함께 보기 때문에, **알 수 없는 단어라도 그 주변 문맥을 통해 의미를 추론**할 수 있다.

→ 따라서, OOV 단어를 완전히 배제하지 않고, 대신 **특수 토큰** `<unk>`(unknown)으로 대체해 처리한다.

---

### `<unk>` 토큰의 역할

- 어휘 사전에 없는 모든 단어는 `<unk>` 토큰으로 대체됨
- 이 토큰은 **고유하게만 설정**되어 있다면 이름은 꼭 `<unk>`일 필요는 없음  
  - 예: `<unknown>`, `[UNK]`, `@@unk@@` 등

---

이렇게 하면 **모델이 학습되지 않은 단어**가 등장하더라도 일관된 방식으로 처리할 수 있으며, 문맥 기반으로 충분히 예측이 가능해진다.


In [ ]:
# Add the unknown token to our vocabulary
vocabulary.add("<unk>")

### 왜 Word Window Classification인가?

우리는 앞서 이 과제가 **Word Window Classification**이라고 불리는 이유에 대해 언급한 적이 있다.  
그 이유는, **모델이 단어를 예측할 때 해당 단어뿐만 아니라 그 주변 단어들까지 함께 고려하기 때문**이다.

---

### 예시 문장: `"We always come to Paris"`

- 정답 레이블: `[0, 0, 0, 0, 1]`  
- 이유: 오직 마지막 단어 `"Paris"`만 **LOCATION (장소명)**이기 때문

---

### 단어 하나만 보면 안 되는 이유

모델이 `"Paris"` 하나만 보고 예측하게 된다면,  
예를 들어 `"to"`라는 단어가 자주 LOCATION 앞에 등장한다는  
**중요한 문맥 정보**를 활용할 수 없다.

→ **그래서 Word Window라는 개념을 도입**한다.

---

### Word Window 개념

- 예측 시, 중심 단어 기준으로 **앞뒤 N개의 단어 (총 2N + 1개)**를 함께 고려
- 윈도우 크기(`window size`)가 1이라면:
  - 입력 단어 수: 앞 1, 중심 1, 뒤 1 → **총 3개의 단어**

#### 예시 (윈도우 크기 = 1):
중심 단어 `"Paris"` → 입력: `["to", "Paris", <pad>]`  
(※ `"Paris"`는 문장의 마지막 단어이므로 뒤 단어가 없음)

---

### 고정된 입력 크기의 문제

PyTorch 모델을 초기화할 때는 입력 크기를 **미리 고정**해야 한다.  
하지만 문장의 처음이나 끝에 위치한 단어들은 **윈도우의 일부가 부족**할 수 있다.

→ 예: `"Paris"`는 마지막 단어이므로 뒤 단어가 없음  
→ 예: `"We"`는 첫 단어이므로 앞 단어가 없음

---

### 해결책: `<pad>` 토큰

- 문장의 앞과 뒤에 `<pad>` 같은 **특수 토큰**을 추가
- 모든 단어가 항상 **고정된 윈도우 크기**를 유지할 수 있도록 보장

이렇게 하면 모델은 **항상 동일한 입력 크기**로 데이터를 받아 처리할 수 있다.


In [ ]:
# Add the <pad> token to our vocabulary
vocabulary.add("<pad>")

# Function that pads the given sentence
# We are introducing this function here as an example
# We will be utilizing it later in the tutorial
def pad_window(sentence, window_size, pad_token="<pad>"):
  window = [pad_token] * window_size
  return window + sentence + window

# Show padding example
window_size = 2
pad_window(train_sentences[0], window_size=window_size)

Now that our vocabularly is ready, let's assign an index to each of our words.

In [ ]:
# We are just converting our vocabularly to a list to be able to index into it
# Sorting is not necessary, we sort to show an ordered word_to_ind dictionary
# That being said, we will see that having the index for the padding token
# be 0 is convenient as some PyTorch functions use it as a default value
# such as nn.utils.rnn.pad_sequence, which we will cover in a bit
# <pad>의 index값은 0
ix_to_word = sorted(list(vocabulary))

# Creating a dictionary to find the index of a given word
word_to_ix = {word: ind for ind, word in enumerate(ix_to_word)}
word_to_ix

In [ ]:
ix_to_word[1]

Great! We are ready to convert our training sentences into a sequence of indices corresponding to each token.

In [ ]:
# Given a sentence of tokens, return the corresponding indices
def convert_token_to_indices(sentence, word_to_ix):
  indices = []
  for token in sentence:
    # Check if the token is in our vocabularly. If it is, get it's index.
    # If not, get the index for the unknown token.
    if token in word_to_ix:
      index = word_to_ix[token]
    else:
      index = word_to_ix["<unk>"] # 단어가 없을 경우
    indices.append(index)
  return indices

# More compact version of the same function
def _convert_token_to_indices(sentence, word_to_ix):
  return [word_to_ind.get(token, word_to_ix["<unk>"]) for token in sentence]

# Show an example
example_sentence = ["we", "always", "come", "to", "kuwait"]
example_indices = convert_token_to_indices(example_sentence, word_to_ix)
restored_example = [ix_to_word[ind] for ind in example_indices]

print(f"Original sentence is: {example_sentence}")
print(f"Going from words to indices: {example_indices}")
print(f"Going from indices to words: {restored_example}")

In the example above, `kuwait` shows up as `<unk>`, because it is not included in our vocabulary. Let's convert our `train_sentences` to `example_padded_indices`.

In [ ]:
# Converting our sentences to indices
example_padded_indices = [convert_token_to_indices(s, word_to_ix) for s in train_sentences]
example_padded_indices

이제 각 단어에 대한 인덱스가 있으므로, PyTorch의 `nn.Embedding` 클래스를 사용하여 임베딩 테이블을 만들 수 있다. 이는 `nn.Embedding(num_words, embedding_dimension)`로 호출되며, 여기서 `num_words`는 어휘의 단어 수이고 `embedding_dimension`은 원하는 임베딩 차원이다. `nn.Embedding`에는 특별한 점이 없다: 이는 단지 학습 가능한 `NxE` 차원 텐서를 감싸는 래퍼 클래스일 뿐이다. 여기서 `N`은 어휘의 단어 수이고 `E`는 임베딩 차원 수이다. 이 테이블은 처음에는 무작위이지만, 시간이 지나면서 변할 것이다. 네트워크를 훈련시키면서 기울기가 임베딩 레이어까지 역전파되어 단어 임베딩이 업데이트된다. 모델에서 사용할 임베딩 레이어를 모델에서 초기화할 것이지만, 여기서 예제를 보여준다.

In [ ]:
# Creating an embedding table for our words
embedding_dim = 5
embeds = nn.Embedding(len(vocabulary), embedding_dim)

# Printing the parameters in our embedding table
list(embeds.parameters())

어휘에 있는 단어의 임베딩을 얻으려면 조회 텐서를 만들기만 하면 된다. 조회 텐서는 우리가 조회하려는 인덱스를 포함하는 텐서일 뿐이다. `nn.Embedding` 클래스는 Long Tensor 타입의 인덱스 텐서를 기대하므로, 이에 맞춰 텐서를 만들어야 한다.

In [ ]:
# Get the embedding for the word Paris
index = word_to_ix["paris"] # 숫자 변환 
index_tensor = torch.tensor(index, dtype=torch.long) # tesor를 필요로 함 
paris_embed = embeds(index_tensor)
paris_embed

In [ ]:
# We can also get multiple embeddings at once
index_paris = word_to_ix["paris"]
index_ankara = word_to_ix["ankara"]
indices = [index_paris, index_ankara]
indices_tensor = torch.tensor(indices, dtype=torch.long)
embeddings = embeds(indices_tensor)
embeddings

Usually, we define the embedding layer as part of our model, which you will see in the later sections of our notebook.

### 학습 방식: 한 문장씩이 아닌 "배치 단위"로 학습하기

지금까지는 모델 학습을 **"한 문장씩" 처리**하는 방식으로 진행했다.  
하지만 실제 딥러닝에서는 보통 다음의 세 가지 방식 중 하나를 선택한다.

---

### 1. 전체 데이터를 한꺼번에 처리 (Batch Gradient Descent)

- **모든 데이터를 사용해 한 번에 평균 손실(loss)**을 구하고 파라미터 업데이트
- **장점**: 가장 **안정적인 경사(gradient)** 계산 가능  
- **단점**: 데이터가 많을 경우 **속도가 느리고, GPU 메모리 초과** 가능성 있음

---

### 2. 데이터 하나씩 처리 (Stochastic Gradient Descent, SGD)

- **한 문장, 한 예제마다** 파라미터를 바로 업데이트  
- **장점**: 빠르게 학습 시작 가능  
- **단점**: **계산이 불안정**, 손실이 크게 출렁일 수 있음

---

### 3. 적절한 해결책: 미니배치 (Mini-batch Gradient Descent)

- 여러 개의 예제를 묶어 **"배치(batch)" 단위로 학습**
- **장점**:
  - 전체보다는 **빠르고**
  - 개별보다 **안정적인** 경사 계산 가능  
- **PyTorch에서는 `DataLoader`**라는 도구를 통해  
  **미니배치 학습을 쉽게 구현**할 수 있음

---

💡 결론: **속도와 안정성의 균형**을 위해 대부분의 딥러닝 모델은 **Mini-batch 방식**을 사용한다.


In [ ]:
from torch.utils.data import DataLoader
from functools import partial

def custom_collate_fn(batch, window_size, word_to_ix):
  # Break our batch into the training examples (x) and labels (y)
  # We are turning our x and y into tensors because nn.utils.rnn.pad_sequence
  # method expects tensors. This is also useful since our model will be
  # expecting tensor inputs.
  x, y = zip(*batch) # 배치를 입력과 라벨로 분류 (sentence,label)형태의 튜플일 것이기 때문에

  # Now we need to window pad our training examples. We have already defined a
  # function to handle window padding. We are including it here again so that
  # everything is in one place.
  def pad_window(sentence, window_size, pad_token="<pad>"):
    window = [pad_token] * window_size
    return window + sentence + window

  # Pad the train examples.
  x = [pad_window(s, window_size=window_size) for s in x]

  # Now we need to turn words in our training examples to indices. We are
  # copying the function defined earlier for the same reason as above.
  def convert_tokens_to_indices(sentence, word_to_ix):
    return [word_to_ix.get(token, word_to_ix["<unk>"]) for token in sentence]

  # Convert the train examples into indices.
  x = [convert_tokens_to_indices(s, word_to_ix) for s in x]

  # We will now pad the examples so that the lengths of all the example in
  # one batch are the same, making it possible to do matrix operations.
  # We set the batch_first parameter to True so that the returned matrix has
  # the batch as the first dimension.
  pad_token_ix = word_to_ix["<pad>"]

  # pad_sequence function expects the input to be a tensor, so we turn x into one
  x = [torch.LongTensor(x_i) for x_i in x]
  x_padded = nn.utils.rnn.pad_sequence(x, batch_first=True, padding_value=pad_token_ix) # 입력으로 써야하기에 문장 길이를 동일하게 맞춰줌

  # We will also pad the labels. Before doing so, we will record the number
  # of labels so that we know how many words existed in each example.
  lengths = [len(label) for label in y]
  lenghts = torch.LongTensor(lengths) 
# 각 단어마다 라벨링이 되기에 <pad>는 0으로 라벨링해서 길이 맞추기
  y = [torch.LongTensor(y_i) for y_i in y]
  y_padded = nn.utils.rnn.pad_sequence(y, batch_first=True, padding_value=0)

  # We are now ready to return our variables. The order we return our variables
  # here will match the order we read them in our training loop.
  return x_padded, y_padded, lenghts

This function seems long, but it really doesn't have to be. Check out the alternative version below where we remove the extra function declarations and comments.

In [ ]:
def _custom_collate_fn(batch, window_size, word_to_ix):
  # Prepare the datapoints
  x, y = zip(*batch)
  x = [pad_window(s, window_size=window_size) for s in x]
  x = [convert_tokens_to_indices(s, word_to_ix) for s in x]

  # Pad x so that all the examples in the batch have the same size
  pad_token_ix = word_to_ix["<pad>"]
  x = [torch.LongTensor(x_i) for x_i in x]
  x_padded = nn.utils.rnn.pad_sequence(x, batch_first=True, padding_value=pad_token_ix)

  # Pad y and record the length
  lengths = [len(label) for label in y]
  lenghts = torch.LongTensor(lengths)
  y = [torch.LongTensor(y_i) for y_i in y]
  y_padded = nn.utils.rnn.pad_sequence(y, batch_first=True, padding_value=0)

  return x_padded, y_padded, lenghts

### `torch.utils.data.DataLoader`란?

`DataLoader`는 PyTorch에서 **데이터를 배치 단위로 자동으로 불러오고**,  
**모델에 넣기 편하게 구성**해주는 유틸리티 클래스이다.

---

### 주요 기능

- **배치 단위(batch-wise)**로 데이터 불러오기  
- **셔플(shuffling)** 기능 지원 → 데이터 순서를 무작위로 섞어 일반화 향상  
- **병렬 처리(num_workers)**로 데이터 로딩 속도 향상 가능  
- **이터레이터 형태**로 동작 → `for batch in dataloader:`처럼 사용 가능

---

### 왜 필요한가?

학습 시 데이터를 직접 한 줄씩 불러오는 것은 번거롭고 비효율적이다.  
`DataLoader`는 이를 자동화하여 **모델 학습을 더 효율적이고 안정적으로** 만들어 준다

---

### 요약

> ✅ `DataLoader`는 PyTorch에서 **모델 학습을 위한 데이터 관리와 배치 처리**를 도와주는 핵심 도구이다.


In [ ]:
# Parameters to be passed to the DataLoader
data = list(zip(train_sentences, train_labels))
batch_size = 2
shuffle = True
window_size = 2
collate_fn = partial(custom_collate_fn, window_size=window_size, word_to_ix=word_to_ix)

# Instantiate the DataLoader
loader = DataLoader(data, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

# Go through one loop
counter = 0
for batched_x, batched_y, batched_lengths in loader:
  print(f"Iteration {counter}")
  print("Batched Input:")
  print(batched_x)
  print("Batched Labels:")
  print(batched_y)
  print("Batched Lengths:")
  print(batched_lengths)
  print("")
  counter += 1

### 모델 입력: 윈도우 기반 분류기를 위한 전처리 완료

앞서 만든 배치 입력 텐서들은 이제 **모델에 전달될 준비**가 되었다.  
우리가 만들 모델은 **윈도우 기반 분류기(Window Classifier)**이다.

---

### 데이터 구조와 예측 방식

- 현재 입력된 텐서들은 **한 문장의 모든 단어가 하나의 데이터 포인트로 묶인 형태**이다.
- 모델은 이 데이터를 입력으로 받아 **각 단어를 중심으로 윈도우(window)**를 만들고,
- **중심 단어가 LOCATION인지 아닌지를 예측**해야 한다.
- 예측 결과는 **전체 문장에 대한 위치 정보(예: [0, 0, 0, 1])**로 반환된다.

---

### 윈도우 단위로 데이터를 미리 자를 수도 있지만...

- 데이터를 **사전에 윈도우 단위로 잘라 모델에 넣는 방식도 가능**하지만,  
- 이런 윈도우 생성 과정을 **모델 내부에서 처리**할 수도 있다.  
- 이 방식은 **모델을 더 일반적이고 유연하게** 만들어 준다.

---

### 윈도우 크기와 예측 횟수

- 윈도우 크기를 `N`이라고 하면, 매번 **2N + 1개의 단어**를 보고 예측을 수행
- 예시: 입력로 9개의 단어, 윈도우 크기 2 → 총 **5개의 예측 수행**

---

### 윈도우 생성: `unfold()` 메서드 활용

PyTorch에서는 **효율적으로 윈도우를 생성**할 수 있는 메서드가 존재한다:

```python
tensor.unfold(dimension, size, step)


In [ ]:
# Print the original tensor
print(f"Original Tensor: ")
print(batched_x)
print("")

# Create the 2 * 2 + 1 chunks
chunk = batched_x.unfold(1, window_size*2 + 1, 1)
print(f"Windows: ")
print(chunk)

### Model

Now that we have prepared our data, we are ready to build our model. We have learned how to write custom `nn.Module` classes. We will do the same here and put everything we have learned so far together.

In [ ]:
class WordWindowClassifier(nn.Module):

  def __init__(self, hyperparameters, vocab_size, pad_ix=0):
    super(WordWindowClassifier, self).__init__()

    """ Instance variables """
    self.window_size = hyperparameters["window_size"]
    self.embed_dim = hyperparameters["embed_dim"]
    self.hidden_dim = hyperparameters["hidden_dim"]
    self.freeze_embeddings = hyperparameters["freeze_embeddings"]

    """ Embedding Layer
    Takes in a tensor containing embedding indices, and returns the
    corresponding embeddings. The output is of dim
    (number_of_indices * embedding_dim).

    If freeze_embeddings is True, set the embedding layer parameters to be
    non-trainable. This is useful if we only want the parameters other than the
    embeddings parameters to change.

    """
    self.embeds = nn.Embedding(vocab_size, self.embed_dim, padding_idx=pad_ix)
    if self.freeze_embeddings:
      self.embed_layer.weight.requires_grad = False

    """ Hidden Layer
    """
    full_window_size = 2 * window_size + 1
    self.hidden_layer = nn.Sequential(
      nn.Linear(full_window_size * self.embed_dim, self.hidden_dim),
      nn.Tanh()
    )

    """ Output Layer
    """
    self.output_layer = nn.Linear(self.hidden_dim, 1)

    """ Probabilities
    """
    self.probabilities = nn.Sigmoid()

  def forward(self, inputs):
    """
    Let B:= batch_size
        L:= window-padded sentence length
        D:= self.embed_dim
        S:= self.window_size
        H:= self.hidden_dim

    inputs: a (B, L) tensor of token indices
    """
    B, L = inputs.size()

    """
    Reshaping.
    Takes in a (B, L) LongTensor
    Outputs a (B, L~, S) LongTensor
    """
    # Fist, get our word windows for each word in our input.
    token_windows = inputs.unfold(1, 2 * self.window_size + 1, 1)
    _, adjusted_length, _ = token_windows.size()

    # Good idea to do internal tensor-size sanity checks, at the least in comments!
    assert token_windows.size() == (B, adjusted_length, 2 * self.window_size + 1)

    """
    Embedding.
    Takes in a torch.LongTensor of size (B, L~, S)
    Outputs a (B, L~, S, D) FloatTensor.
    """
    embedded_windows = self.embeds(token_windows)

    """
    Reshaping.
    Takes in a (B, L~, S, D) FloatTensor.
    Resizes it into a (B, L~, S*D) FloatTensor.
    -1 argument "infers" what the last dimension should be based on leftover axes.
    """
    # nn.Linear에 넣기 위한 평탄화 작업 n*m(행렬)-> n*m(벡터) 
    embedded_windows = embedded_windows.view(B, adjusted_length, -1)

    """
    Layer 1.
    Takes in a (B, L~, S*D) FloatTensor.
    Resizes it into a (B, L~, H) FloatTensor
    """
    layer_1 = self.hidden_layer(embedded_windows)

    """
    Layer 2
    Takes in a (B, L~, H) FloatTensor.
    Resizes it into a (B, L~, 1) FloatTensor.
    """
    output = self.output_layer(layer_1)

    """
    Softmax.
    Takes in a (B, L~, 1) FloatTensor of unnormalized class scores.
    Outputs a (B, L~, 1) FloatTensor of (log-)normalized class scores.
    """
    output = self.probabilities(output)
    output = output.view(B, -1)

    return output

### 임베딩 벡터 평탄화(Flatten)란?
여러 개의 임베딩 벡터를 한 개의 긴 벡터로 이어 붙이는 것

### Training

We are now ready to put everything together. Let's start with preparing our data and intializing our model. We can then intialize our optimizer and define our loss function. This time, instead of using one of the predefined loss function as we did before, we will define our own loss function.

In [ ]:
# Prepare the data
data = list(zip(train_sentences, train_labels))
batch_size = 2
shuffle = True
window_size = 2
collate_fn = partial(custom_collate_fn, window_size=window_size, word_to_ix=word_to_ix)

# Instantiate a DataLoader
loader = DataLoader(data, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

# Initialize a model
# It is useful to put all the model hyperparameters in a dictionary
model_hyperparameters = {
    "batch_size": 4,
    "window_size": 2,
    "embed_dim": 25,
    "hidden_dim": 25,
    "freeze_embeddings": False,
}

vocab_size = len(word_to_ix)
model = WordWindowClassifier(model_hyperparameters, vocab_size)

# Define an optimizer
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Define a loss function, which computes to binary cross entropy loss
def loss_function(batch_outputs, batch_labels, batch_lengths):
    # Calculate the loss for the whole batch
    bceloss = nn.BCELoss()
    loss = bceloss(batch_outputs, batch_labels.float())

    # Rescale the loss. Remember that we have used lengths to store the
    # number of words in each training example
    loss = loss / batch_lengths.sum().float()

    return loss

Unlike our earlier example, this time instead of passing all of our training data to the model at once in each epoch, we will be utilizing batches. Hence, in each training epoch iteration, we also iterate over the batches.

In [ ]:
# Function that will be called in every epoch
def train_epoch(loss_function, optimizer, model, loader):

  # Keep track of the total loss for the batch
  total_loss = 0
  for batch_inputs, batch_labels, batch_lengths in loader:
    # Clear the gradients
    optimizer.zero_grad()
    # Run a forward pass
    outputs = model.forward(batch_inputs)
    # Compute the batch loss
    loss = loss_function(outputs, batch_labels, batch_lengths)
    # Calculate the gradients
    loss.backward()
    # Update the parameteres
    optimizer.step()
    total_loss += loss.item()

  return total_loss


# Function containing our main training loop
def train(loss_function, optimizer, model, loader, num_epochs=10000):

  # Iterate through each epoch and call our train_epoch function
  for epoch in range(num_epochs):
    epoch_loss = train_epoch(loss_function, optimizer, model, loader)
    if epoch % 100 == 0: print(epoch_loss)

Let's start training!

In [ ]:
num_epochs = 1000
train(loss_function, optimizer, model, loader, num_epochs=num_epochs)

### Prediction

Let's see how well our model is at making predictions. We can start by creating our test data.

In [ ]:
# Create test sentences
test_corpus = ["She comes from Paris"]
test_sentences = [s.lower().split() for s in test_corpus]
test_labels = [[0, 0, 0, 1]]

# Create a test loader
test_data = list(zip(test_sentences, test_labels))
batch_size = 1
shuffle = False
window_size = 2
collate_fn = partial(custom_collate_fn, window_size=2, word_to_ix=word_to_ix)
test_loader = torch.utils.data.DataLoader(test_data,
                                           batch_size=1,
                                           shuffle=False,
                                           collate_fn=collate_fn)

Let's loop over our test examples to see how well we are doing.

In [ ]:
for test_instance, labels, _ in test_loader:
  outputs = model.forward(test_instance)
  print(labels)
  print(outputs)